In [10]:
from dataclasses import dataclass,field
import os.path
from os import listdir
import sys
from enum import Enum
import importlib
import json
import jsons
import math
import requests
import datetime
import gzip

from collections import OrderedDict

#Change input_directory to Elite Insight log directory
input_directory = 'C:\\GW2Logs\\Output\\'
files = listdir(input_directory)
sorted_files = sorted(files)
test_text=""
players_running_healing_addon=[]
Fight_Review={}

for filename in sorted_files:
    
    file_start, file_extension = os.path.splitext(filename)
    # skip files of incorrect filetype
    if file_extension not in ['.json', '.gz']:
        continue
    #if filename not in ['TW5_top_stats_202507151431.json']:
        #continue
    file_path = "".join((input_directory,"/",filename))

    if file_extension == '.gz':
        with gzip.open(file_path, mode="r") as f:
            json_data = json.loads(f.read().decode('utf-8'))
    else:
        json_datafile = open(file_path, encoding='utf-8')
        json_data = json.load(json_datafile)    

    print(f"Processing file: {file_start}")
    
    if 'usedExtensions' not in json_data:
        players_running_healing_addon = []
    else:
        extensions = json_data['usedExtensions']
        for extension in extensions:
            if extension['name'] == "Healing Stats":
                #players_running_healing_addon = extension['runningExtension']
                for healer_name in extension['runningExtension']:
                    if healer_name not in players_running_healing_addon:
                        players_running_healing_addon.append(healer_name)
                
    if file_start not in Fight_Review:
        Fight_Review[file_start] = {}
    fight_min = 0
    fight_max = 0
    players = json_data['players']                

    for player in players:
        if player['notInSquad']:
            continue
        print(f"processing player: {player['name']}")
        player_group = player['group']
        player_prof = player['profession']
        player_name = player['name']
        player_acct = player['account']
        player_key = f"{player_name}|{player_prof}|{player_acct}|{player_group}"
        damage_taken1S = player['damageTaken1S']
        health_pct = player['healthPercents']
        barrier_pct = player['barrierPercents']

        if player_key not in Fight_Review[file_start]:
            Fight_Review[file_start][player_key] = {
                "group": player_group,
                "name": player_name,
                "prof": player_prof,
                "damage_taken1S": damage_taken1S,
                "health_pct": health_pct,
                "barrier_pct": barrier_pct
            }

print('---=====Complete=====---')



Processing file: 20250908-204222_detailed_wvw_kill.json
processing player: Ami Lightbringer
processing player: Baebel
processing player: Beep Bop Praetor
processing player: Lord Elex
processing player: Qtpi Gluvs
processing player: Caelon Cai
processing player: Drevarr Moonwillow
processing player: Moon In Pink
processing player: Reppalskay
processing player: Xanvias Tsolice
processing player: Ayame Kuromori
processing player: Komyni
processing player: Muxi W
processing player: Piero Rosso
processing player: Silvyrs
processing player: Amaresu Lan
processing player: Badbluntnotgud
processing player: Huskymirale Builder
processing player: Newtype Clan
processing player: Rainy Rainy Go Away
---=====Complete=====---


In [13]:
Fight_Review['20250908-204222_detailed_wvw_kill.json']['Ami Lightbringer|Firebrand|Amitiel.9436|2']

{'group': 2,
 'name': 'Ami Lightbringer',
 'prof': 'Firebrand',
 'damage_taken1S': [[0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   105,
   411,
   914,
   1102,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1298,
   1795,
   2294,
   2294,
   12052,
   28086,
   31185,
   31185,
   31185,
   32359,
   32359,
   32359,
   32388,
   32388,
   32388,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   32706,
   33297,
   33918,
   33918,
   339

In [76]:
Regen_Healing['Drevarr Moonwillow']

{'Drevarr Moonwillow': {'healing': 25536, 'hits': 70},
 'Reppalskay': {'healing': 1352, 'hits': 2}}

In [97]:
header = "|Heal Target |"
for name in players_running_healing_addon:
    header+=f" !{name[:10]}|"
header+="h"
print(header)
line=""
for player in Regen_Healing:
    line+=f"|{player} |"
    for healer in players_running_healing_addon:
        if healer in Regen_Healing[player]:
            line+=f" {Regen_Healing[player][healer]['healing']:,.0f}|"
        else:
            line+=f" |"
    line+="\n"
print(line)

|Heal Target | !Drevarr Mo| !Serafina E| !Silvyrs| !Mac Ottere| !Praetorian| !Reppalskay| !Chocolate | !Amalgam Ca| !Ms Elitia| !Upper D Am| !Adapted In| !Corrupted | !Hanbee Ele| !Mistwarden| !Komyni| !Hang L| !Kanta Kahn| !Arcana In | !Relexis| !Elemental | !Gandalffa |h
|Amitiels Revenge | 54,980| 954| | 72,390| 171| 11,202| 17,232| | 2,396| | | 398| | | | 1,746| | | | | |
|Moon In Pink | 65,984| | 770| 49,041| | 1,517| 16,535| | 1,528| | | 259| 967| | | | | | | | |
|Ms Elitia | 67,775| | 631| 55,042| | | 5,684| | 1,565| | | 1,350| | | | | | | | | |
|Drevarr Moonwillow | 97,296| 531| 269| 2,289| 178| 676| 1,303| | 154| | | | | | | | | | | | |
|Muxi W | 82,546| | 323| 7,338| | 2,775| 10,557| | | | | | 283| | | | | | | | |
|Newtype Clan | 94,319| | 622| 5,277| | | 9,777| | | | | 162| | | | 445| | | | | |
|Aezlenne | 22,155| | | 12,176| | 13,002| 65,640| | | | | 944| 714| | | 2,242| | | | | |
|Reppalskay | 37,827| 1,095| 388| 13,315| | 8,228| 57,991| | 307| | 197| 29| 488| | | 476| | |

In [88]:
json_datafile.close()